# Imports and configuration

In [18]:

from collections import Counter
from datasets import load_dataset

from transformers import AutoTokenizer
from datasets import load_dataset, DatasetDict

MODEL_NAMES = ["roberta-base", "distilroberta-base"]
MAX_LENGTH = 128



# Dataset loading and exploration

In [13]:
# Load the QQP configuration of the GLUE benchmark.
qqp = load_dataset("nyu-mll/glue", "qqp")

# Inspect available splits and their sizes.
for name, split in qqp.items():
    print(f"{name}: {len(split):,} examples")

print("\nFeatures:")
print(qqp["train"].features)

print("\nSample training example:")
for i in range(5):
    print(qqp["train"][i])

#print(qqp["train"][0])

# Inspect labels and missing question text.
for name in ("train", "validation"):
    split = qqp[name]
    print(f"\n{name} label distribution:")
    print(Counter(split["label"]))

    missing = sum(
        not (q1 or "").strip() or not (q2 or "").strip()
        for q1, q2 in zip(split["question1"], split["question2"])
    )
    print(f"Missing/empty pairs: {missing}")


train: 363,846 examples
validation: 40,430 examples
test: 390,965 examples

Features:
{'question1': Value('string'), 'question2': Value('string'), 'label': ClassLabel(names=['not_duplicate', 'duplicate']), 'idx': Value('int32')}

Sample training example:
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}
{'question1': 'How do I control my horny emotions?', 'question2': 'How do you control your horniness?', 'label': 1, 'idx': 1}
{'question1': 'What causes stool color to change to yellow?', 'question2': 'What can cause stool to come out as little balls?', 'label': 0, 'idx': 2}
{'question1': 'What can one do after MBBS?', 'question2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3}
{'question1': 'Where can I find a power outlet for my laptop at Melbourne Airport?', 'question2': 'Would a second airport in Sydney, Australia be needed if a high-speed r

# Tokenization (Question Pair)

Converts human-readable questions into numerical tokens the Transformer can process.

In [19]:

def tokenize_pairs(batch, tokenizer):
    """Tokenize question pairs for Transformer classification."""
    return tokenizer(
        batch["question1"],
        batch["question2"],
        truncation=True,
        max_length=MAX_LENGTH
    )

# Store each model's tokenizer so we can reuse it later.
# Different pretrained models may require different tokenizers.
tokenizers = {}

# Select only 5 training examples for a quick smoke test.
# This avoids processing the entire dataset before verifying our code.
sample = qqp["train"].select(range(5))

# Run the same tokenization test for both models.
for model_name in MODEL_NAMES:

    # Load the tokenizer associated with the pretrained model.
    # It converts text into numerical token IDs understood by the model.
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Keep the tokenizer for later training and inference.
    tokenizers[model_name] = tokenizer

    # Tokenize both questions together as a sequence pair.
    # Output includes input_ids and attention_mask.
    encoded = tokenize_pairs(sample[:], tokenizer)

    # Verify that all 5 question pairs were successfully tokenized.
    assert len(encoded["input_ids"]) == 5

    # Check that no encoded pair exceeds the maximum sequence length.
    # MAX_LENGTH applies to BOTH questions combined, including special tokens.
    assert all(
        len(ids) <= MAX_LENGTH
        for ids in encoded["input_ids"]
    )

    # Inspect the first encoded example to understand the output.
    print(f"\nModel: {model_name}")
    print(f"Token count: {len(encoded['input_ids'][0])}")

    # Display the first 15 numerical token IDs.
    print(f"Input IDs: {encoded['input_ids'][0][:15]}")

    # Convert token IDs back into readable text.
    # Useful for checking how the two questions were combined.
    print(f"Decoded: {tokenizer.decode(encoded['input_ids'][0])}")

    # If assertions did not fail, the basic tokenization checks passed.
    print("PASSED")


# Store the tokenized datasets separately for each model.
# Both models will use the SAME original training and validation data.
tokenized_datasets = {}

for model_name in MODEL_NAMES:

    print(f"\nTokenizing dataset for {model_name}...")

    # Reuse the tokenizer loaded during the smoke test.
    tokenizer = tokenizers[model_name]

    # Keep only the splits needed for training and evaluation.
    # The official QQP test set does not contain public ground-truth labels.
    dataset = DatasetDict({
        "train": qqp["train"],
        "validation": qqp["validation"]
    })

    # Convert all question pairs into numerical tokens.
    # batched=True processes multiple examples per function call.
    # remove_columns removes original text and idx from the
    # tokenized copy, but preserves the label for training.
    tokenized = dataset.map(
        tokenize_pairs,
        batched=True,
        fn_kwargs={"tokenizer": tokenizer},
        remove_columns=["question1", "question2", "idx"]
    )

    # Save the processed dataset for this model.
    tokenized_datasets[model_name] = tokenized

    print(f"Train: {len(tokenized['train']):,}")
    print(f"Validation: {len(tokenized['validation']):,}")
    print(f"Features: {tokenized['train'].column_names}")



Model: roberta-base
Token count: 36
Input IDs: [0, 6179, 16, 5, 301, 9, 10, 10638, 1294, 116, 9918, 47, 6190, 110, 308]
Decoded: <s>How is the life of a math student? Could you describe your own experiences?</s></s>Which level of prepration is enough for the exam jlpt5?</s>
PASSED

Model: distilroberta-base
Token count: 36
Input IDs: [0, 6179, 16, 5, 301, 9, 10, 10638, 1294, 116, 9918, 47, 6190, 110, 308]
Decoded: <s>How is the life of a math student? Could you describe your own experiences?</s></s>Which level of prepration is enough for the exam jlpt5?</s>
PASSED

Tokenizing dataset for roberta-base...


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Train: 363,846
Validation: 40,430
Features: ['label', 'input_ids', 'attention_mask']

Tokenizing dataset for distilroberta-base...


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Train: 363,846
Validation: 40,430
Features: ['label', 'input_ids', 'attention_mask']


In [20]:
for model_name in MODEL_NAMES:

    dataset = tokenized_datasets[model_name]

    # Tokenization must not change the number of examples.
    assert len(dataset["train"]) == 363846
    assert len(dataset["validation"]) == 40430

    # Confirm the model input features and classification labels exist.
    required_columns = {"input_ids", "attention_mask", "label"}

    assert required_columns.issubset(
        dataset["train"].column_names
    )

    # Verify sequence lengths on the first 100 examples.
    assert all(
        len(ids) <= MAX_LENGTH
        for ids in dataset["train"][:100]["input_ids"]
    )

    print(f"{model_name}: PASSED")

print("\nFull dataset tokenization verified.")

roberta-base: PASSED
distilroberta-base: PASSED

Full dataset tokenization verified.


# Model setup

# Fine-tuning

# Evaluation

# Comparison

# Custom predictions

Test at least five examples

# Error analysis

Examine incorrect classifications

# Conclusions and export